# KIC 공모전: 산업·사업모델 분석 + 기술적 분석 에이전트 (주피터 노트북판)

이 노트북 하나로 두 에이전트를 만들고 바로 실행해 볼 수 있어요.

- **1부 (설치)**: 아래 `%%writefile` 셀들이 노트북 옆에 `kic_agents/` 폴더를 만들어요. 코드를 고치고 싶으면 해당 셀을 고친 뒤 다시 실행하면 돼요.
- **2부 (실행)**: 페르소나, 기술적 분석, 산업·사업모델 분석, Orchestrator, Weight 랭킹을 차례로 돌려 봐요.

원칙: **숫자는 코드가 계산하고, Claude는 문장만 씁니다.** API 키가 없으면 템플릿 문장으로 끝까지 동작해요.
데이터는 기본이 **합성 샘플**입니다(출처 type `SAMPLE`). 실제 시세는 맨 아래 pykrx 셀을 쓰세요.

설계서: https://claude.ai/artifact/7YkEtpmZDUJa49gUeZpeSm

**실행 방법**: 메뉴에서 *Run All* (또는 위에서부터 Shift+Enter).

## 0. 준비

In [ ]:
# 처음 한 번만 실행하세요 (이미 있으면 건너뜀)
%pip install -q anthropic pandas numpy pydantic matplotlib
# 실제 시세를 쓰려면: %pip install -q pykrx

In [ ]:
import os, getpass

# Claude로 문장을 쓰려면 API 키를 넣으세요. 비워 두면 템플릿 문장으로 동작합니다.
if not os.getenv("ANTHROPIC_API_KEY") and os.getenv("KIC_USE_LLM", "1") != "0":
    try:
        key = getpass.getpass("ANTHROPIC_API_KEY (없으면 Enter): ").strip()
    except Exception:  # 입력창을 지원하지 않는 환경
        key = ""
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key

os.environ.setdefault("KIC_USE_LLM", "1")        # "0"이면 키가 있어도 LLM을 안 부름 (시연 백업)
os.environ.setdefault("KIC_MODEL", "claude-opus-5-5")
print("LLM 사용:", bool(os.getenv("ANTHROPIC_API_KEY")) and os.environ["KIC_USE_LLM"] != "0")

## 1부. 에이전트 코드 만들기

각 셀이 파일 하나를 만듭니다. 순서대로 실행하세요.

In [ ]:
import os
for d in ["kic_agents", "kic_agents/data", "kic_agents/tools", "kic_agents/agents"]:
    os.makedirs(d, exist_ok=True)
for d in ["kic_agents/data", "kic_agents/tools", "kic_agents/agents"]:
    open(os.path.join(d, "__init__.py"), "a").close()
print("폴더 준비 완료:", os.path.abspath("kic_agents"))

### `kic_agents/__init__.py`
패키지 진입점

In [ ]:
%%writefile kic_agents/__init__.py
"""KIC AI Agent 공모전: 산업·사업모델 분석 에이전트와 기술적 분석 에이전트."""

from kic_agents.schemas import AgentRequest
from kic_agents.personas import get_persona, classify_persona

__all__ = ["AgentRequest", "get_persona", "classify_persona"]

### `kic_agents/schemas.py`
요청·응답 규격 (Orchestrator와 약속한 JSON)

In [ ]:
%%writefile kic_agents/schemas.py
"""에이전트 요청·응답 규격 (설계서 5-6, 6-6, 7장).

두 에이전트가 공통으로 쓰는 필드: agent, ticker, as_of, persona_id, summary,
sources, limitations, factor_scores. Orchestrator는 이 이름만 믿고 합친다.
"""

from __future__ import annotations

from datetime import date
from typing import Any, Literal

from pydantic import BaseModel, Field


class AgentRequest(BaseModel):
    ticker: str = Field(description="6자리 종목코드, 예: 000660")
    question: str = ""
    persona_id: str = "L2-CON"
    weights: dict[str, float] | None = None
    horizon: Literal["short", "long"] | None = None
    as_of: date | None = None


class Source(BaseModel):
    id: str
    type: str  # KIS, KRX, DART, PUBLIC_DATA, SAMPLE ...
    title: str
    date: str | None = None
    url: str | None = None


class Evidence(BaseModel):
    name: str
    value: Any
    source_id: str | None = None


class Item(BaseModel):
    """기회 또는 리스크 한 줄. source_id는 sources 안의 id여야 한다."""

    text: str
    source_id: str


class TermNote(BaseModel):
    term: str
    explanation: str


class BaseResult(BaseModel):
    agent: str
    ticker: str
    name: str = ""
    as_of: str
    persona_id: str
    metrics: list[Evidence] = Field(default_factory=list, description="페르소나에 맞춰 고른 지표")
    factor_scores: dict[str, float] = Field(default_factory=dict, description="0~1, 팩터 점수 엔진 입력")
    opportunities: list[Item] = Field(default_factory=list)
    risks: list[Item] = Field(default_factory=list)
    terms: list[TermNote] = Field(default_factory=list)
    summary: str = ""
    summary_by: Literal["llm", "template"] = "template"
    sources: list[Source] = Field(default_factory=list)
    limitations: list[str] = Field(default_factory=list)


class Phase(BaseModel):
    phase: str
    evidence: list[Evidence] = Field(default_factory=list)


class IndustryResult(BaseResult):
    agent: str = "industry_bm"
    value_chain: dict[str, Any] = Field(default_factory=dict)
    business_model: dict[str, Any] = Field(default_factory=dict)
    cycle: Phase | None = None
    peers: list[dict[str, Any]] = Field(default_factory=list)


class Signal(BaseModel):
    type: str
    date: str
    hit_rate_20d: float | None = None
    avg_return_20d: float | None = None
    n_samples: int = 0
    reliability: Literal["참고용", "보통", "높음"] = "참고용"


class TechnicalResult(BaseResult):
    agent: str = "technical"
    trend: Phase | None = None
    signals: list[Signal] = Field(default_factory=list)
    levels: dict[str, list[float]] = Field(default_factory=dict)
    flows: dict[str, float] = Field(default_factory=dict)
    risk: dict[str, float | None] = Field(default_factory=dict)
    chart_annotations: list[dict[str, str]] = Field(default_factory=list)

### `kic_agents/personas.py`
페르소나 8종, 온보딩 판별(분류 A), 용어 사전

In [ ]:
%%writefile kic_agents/personas.py
"""페르소나 8종 (설계서 1~3장)과 페르소나 분류 (분류 과제 A).

persona_id = "<등급>-<성향>", 예: "L1-AGG", "L3-CON".
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field

LEVELS = ("L1", "L2", "L3", "L4")
STYLES = ("AGG", "CON")

LEVEL_NAMES = {"L1": "입문", "L2": "초급", "L3": "중급", "L4": "고급"}
STYLE_NAMES = {"AGG": "공격형", "CON": "안정형"}

PERSONA_TITLES = {
    "L1-AGG": "테마 추격 입문자",
    "L1-CON": "예금 대신 입문자",
    "L2-AGG": "차트 학습형 단기 투자자",
    "L2-CON": "우량주 찾는 초보",
    "L3-AGG": "성장·모멘텀 투자자",
    "L3-CON": "가치·배당 투자자",
    "L4-AGG": "액티브 팩터 트레이더",
    "L4-CON": "리스크 관리형 자산배분가",
}

# 설계서 3장 설정값 표
_LEVEL_SETTINGS = {
    "L1": dict(max_metrics=2, explain_terms="all", summary_len=(3, 5), show_raw=False),
    "L2": dict(max_metrics=4, explain_terms="first", summary_len=(5, 7), show_raw=False),
    "L3": dict(max_metrics=8, explain_terms="none", summary_len=(7, 10), show_raw=False),
    "L4": dict(max_metrics=99, explain_terms="none", summary_len=(3, 3), show_raw=True),
}

# 팩터 이름은 점수 엔진과 맞춘다: value, growth, profitability, stability, momentum, flow, sentiment
_STYLE_SETTINGS = {
    "AGG": dict(
        emphasis="opportunity_first",
        horizon="short",
        default_weights={"growth": 30, "momentum": 25, "flow": 15, "sentiment": 10, "value": 10, "profitability": 10},
    ),
    "CON": dict(
        emphasis="risk_first",
        horizon="long",
        default_weights={"value": 25, "profitability": 25, "stability": 20, "growth": 10, "momentum": 10, "sentiment": 10},
    ),
}


@dataclass(frozen=True)
class Persona:
    id: str
    level: str
    style: str
    title: str
    max_metrics: int
    explain_terms: str  # all | first | none
    summary_len: tuple[int, int]
    show_raw: bool
    emphasis: str  # opportunity_first | risk_first
    horizon: str  # short | long
    default_weights: dict[str, float] = field(default_factory=dict)

    @property
    def label(self) -> str:
        return f"{LEVEL_NAMES[self.level]}·{STYLE_NAMES[self.style]} ({self.title})"


def get_persona(persona_id: str) -> Persona:
    m = re.fullmatch(r"(L[1-4])-(AGG|CON)", persona_id.strip().upper())
    if not m:
        raise ValueError(f"persona_id 형식이 아닙니다: {persona_id!r} (예: L2-AGG)")
    level, style = m.groups()
    return Persona(
        id=f"{level}-{style}",
        level=level,
        style=style,
        title=PERSONA_TITLES[f"{level}-{style}"],
        **_LEVEL_SETTINGS[level],
        **_STYLE_SETTINGS[style],
    )


def all_personas() -> list[Persona]:
    return [get_persona(f"{lv}-{st}") for lv in LEVELS for st in STYLES]


# ---------- 분류 과제 A: 온보딩 10문항 → persona_id ----------

def classify_literacy(answers: list[int]) -> str:
    """이해도 5문항, 문항당 0~2점 (모름 0 / 대략 앎 1 / 설명 가능 2)."""
    if len(answers) != 5 or any(a not in (0, 1, 2) for a in answers):
        raise ValueError("이해도 답변은 0~2점 5개여야 합니다")
    total = sum(answers)
    if total <= 2:
        return "L1"
    if total <= 5:
        return "L2"
    if total <= 8:
        return "L3"
    return "L4"


def classify_style(answers: list[int]) -> str:
    """성향 5문항, 1~5점 리커트.

    ①하락 시 행동 ②목표 수익률 ③보유 기간 → 성향 점수
    ④투자금 비중(여유 자금) ⑤1년 내 사용 계획 → 여유 자금 점수
    둘 다 3.5 이상이면 공격형. 마음이 공격적이어도 여유 자금이 없으면 안정형.
    """
    if len(answers) != 5 or any(not 1 <= a <= 5 for a in answers):
        raise ValueError("성향 답변은 1~5점 5개여야 합니다")
    appetite = sum(answers[:3]) / 3
    capacity = sum(answers[3:]) / 2
    return "AGG" if appetite >= 3.5 and capacity >= 3.5 else "CON"


def classify_persona(literacy_answers: list[int], style_answers: list[int]) -> str:
    return f"{classify_literacy(literacy_answers)}-{classify_style(style_answers)}"


# 대화 중 재분류: 사용자가 쓴 용어 수준으로 등급을 한 단계 올리거나 내린다.
_ADVANCED_TERMS = ("dcf", "wacc", "ev/ebitda", "백테스트", "샤프", "팩터", "베타", "mdd", "adx", "atr")
_INTERMEDIATE_TERMS = ("per", "pbr", "roe", "rsi", "macd", "이동평균", "영업이익률", "재고자산", "수급")
_BEGINNER_MARKERS = ("뭐예요", "무슨 뜻", "모르겠", "처음", "쉽게")


def adjust_level(persona_id: str, recent_messages: list[str]) -> str:
    p = get_persona(persona_id)
    text = " ".join(recent_messages).lower()
    adv = sum(t in text for t in _ADVANCED_TERMS)
    mid = sum(t in text for t in _INTERMEDIATE_TERMS)
    confused = sum(t in text for t in _BEGINNER_MARKERS)
    idx = LEVELS.index(p.level)
    if confused >= 2 and idx > 0:
        idx -= 1
    elif adv >= 2 and idx < 3:
        idx += 1
    elif mid >= 3 and idx < 2:
        idx += 1
    return f"{LEVELS[idx]}-{p.style}"


GLOSSARY = {
    "이동평균선": "최근 며칠 동안의 평균 주가를 이은 선이에요. 20일선은 한 달, 60일선은 석 달 평균이에요.",
    "RSI": "최근 오른 힘과 내린 힘을 비교한 숫자(0~100)예요. 70 위면 많이 오른 상태, 30 아래면 많이 내린 상태로 봐요.",
    "MACD": "짧은 평균과 긴 평균의 차이로 흐름이 바뀌는지 보는 지표예요.",
    "ADX": "추세가 얼마나 강한지 나타내는 숫자예요. 20~25를 넘으면 뚜렷한 추세로 봐요.",
    "최대낙폭": "고점에서 가장 많이 떨어졌던 비율이에요. 그만큼 떨어질 수 있다는 걸 보여 줘요.",
    "변동성": "주가가 평소에 얼마나 크게 움직이는지예요. 높을수록 오를 때도 내릴 때도 커요.",
    "지지선": "과거에 주가가 내려오다 멈췄던 가격대예요.",
    "저항선": "과거에 주가가 오르다 막혔던 가격대예요.",
    "수급": "누가(외국인·기관·개인) 사고 파는지예요.",
    "매출 성장률": "작년 같은 분기보다 매출이 얼마나 늘었는지예요.",
    "영업이익률": "매출 100원 중 본업으로 남긴 이익이 몇 원인지예요.",
    "부채비율": "자기 돈 대비 빌린 돈의 비율이에요. 100%면 자기 돈만큼 빚이 있다는 뜻이에요.",
    "업황 사이클": "반도체는 수요와 공급이 몇 년 주기로 좋아졌다 나빠졌다를 반복해요. 지금 그 주기의 어디쯤인지예요.",
    "밸류체인": "반도체가 만들어지는 과정(설계→제조→장비·소재→패키징)에서 이 회사가 맡은 자리예요.",
    "CAPEX": "공장·장비에 쓰는 투자 돈이에요. 반도체는 이 돈이 커서 업황이 나쁠 때 부담이 돼요.",
}


def term_notes(persona: Persona, terms: list[str]) -> list[dict[str, str]]:
    """페르소나 설정에 따라 용어 풀이를 붙인다. first는 상위 2개만."""
    if persona.explain_terms == "none":
        return []
    picked = [t for t in terms if t in GLOSSARY]
    if persona.explain_terms == "first":
        picked = picked[:2]
    return [{"term": t, "explanation": GLOSSARY[t]} for t in picked]

### `kic_agents/data/universe.py`
반도체 17종목과 밸류체인 라벨 (분류 B)

In [ ]:
%%writefile kic_agents/data/universe.py
"""반도체 17종목 MVP 범위와 밸류체인 라벨 (분류 과제 B, 수작업 라벨).

종목 목록은 실행계획 2-3의 예시 후보입니다. 팀 확정 후 이 표만 바꾸면 됩니다.
segment 값: 메모리 / 파운드리 / 후공정(OSAT) / 장비 / 소재·부품 / 설계·IP
"""

UNIVERSE: dict[str, dict[str, str]] = {
    "005930": {"name": "삼성전자", "segment": "메모리", "detail": "메모리·파운드리·시스템LSI 종합반도체"},
    "000660": {"name": "SK하이닉스", "segment": "메모리", "detail": "DRAM·NAND, HBM"},
    "042700": {"name": "한미반도체", "segment": "장비", "detail": "후공정 장비(TC 본더)"},
    "000990": {"name": "DB하이텍", "segment": "파운드리", "detail": "8인치 파운드리"},
    "058470": {"name": "리노공업", "segment": "소재·부품", "detail": "테스트 핀·소켓"},
    "095340": {"name": "ISC", "segment": "소재·부품", "detail": "테스트 소켓"},
    "039030": {"name": "이오테크닉스", "segment": "장비", "detail": "레이저 마커·커팅 장비"},
    "403870": {"name": "HPSP", "segment": "장비", "detail": "고압 수소 어닐링 장비"},
    "240810": {"name": "원익IPS", "segment": "장비", "detail": "증착 장비"},
    "036930": {"name": "주성엔지니어링", "segment": "장비", "detail": "증착 장비(ALD/CVD)"},
    "031980": {"name": "피에스케이홀딩스", "segment": "장비", "detail": "후공정 장비(리플로우·디스컴)"},
    "095610": {"name": "테스", "segment": "장비", "detail": "증착·식각 장비"},
    "357780": {"name": "솔브레인", "segment": "소재·부품", "detail": "식각액 등 공정 소재"},
    "005290": {"name": "동진쎄미켐", "segment": "소재·부품", "detail": "포토레지스트 등 공정 소재"},
    "067310": {"name": "하나마이크론", "segment": "후공정(OSAT)", "detail": "패키징·테스트 외주"},
    "064760": {"name": "티씨케이", "segment": "소재·부품", "detail": "SiC 링 등 소모성 부품"},
    "084370": {"name": "유진테크", "segment": "장비", "detail": "증착 장비(LPCVD)"},
}

# 시연용 5종목 (실행계획: 5종목부터 → 17종목)
DEMO_TICKERS = ["005930", "000660", "042700", "058470", "067310"]


def name_of(ticker: str) -> str:
    return UNIVERSE.get(ticker, {}).get("name", ticker)


def segment_of(ticker: str) -> str:
    return UNIVERSE.get(ticker, {}).get("segment", "기타")

### `kic_agents/data/providers.py`
데이터 공급자: 샘플 / pykrx / NEON(PostgreSQL)

In [ ]:
%%writefile kic_agents/data/providers.py
"""데이터 공급자.

에이전트는 DataProvider 인터페이스만 본다. 데이터 담당이 NEON에 적재를 마치면
PostgresProvider로 바꾸기만 하면 되고, 그 전에는 SampleDataProvider(합성 데이터)로
개발·테스트·시연 백업을 한다.

반환 형식
- get_ohlcv     : DataFrame(index=date) [open, high, low, close, volume]
- get_index     : Series(index=date) 지수 종가
- get_flows     : DataFrame(index=date) [foreign, institution, individual] 순매수 금액(원)
- get_financials: DataFrame(분기 오름차순) [quarter, revenue, operating_income, net_income,
                  inventory, capex, total_liabilities, total_equity] (단위: 억원)
- get_exports   : DataFrame(index='YYYY-MM') [value] 반도체 월 수출액(백만 달러)
- get_profile   : dict name, segment, products, customers, top_customer_share, moat
- search_filings: list[dict(text, title, date, url)]
"""

from __future__ import annotations

import hashlib
from abc import ABC, abstractmethod
from datetime import date, timedelta

import numpy as np
import pandas as pd

from kic_agents.data.universe import UNIVERSE, name_of, segment_of
from kic_agents.schemas import Source


class DataProvider(ABC):
    @abstractmethod
    def get_ohlcv(self, ticker: str, end: date, days: int) -> pd.DataFrame: ...

    @abstractmethod
    def get_index(self, code: str, end: date, days: int) -> pd.Series: ...

    @abstractmethod
    def get_flows(self, ticker: str, end: date, days: int) -> pd.DataFrame: ...

    @abstractmethod
    def get_financials(self, ticker: str) -> pd.DataFrame: ...

    @abstractmethod
    def get_exports(self, end: date, months: int) -> pd.DataFrame: ...

    @abstractmethod
    def get_profile(self, ticker: str) -> dict: ...

    @abstractmethod
    def search_filings(self, ticker: str, query: str, k: int = 3) -> list[dict]: ...

    @abstractmethod
    def source(self, kind: str, ticker: str | None = None) -> Source:
        """kind: price | index | flow | financials | exports | profile | filing"""

    def tickers(self) -> list[str]:
        return list(UNIVERSE)


# ---------------------------------------------------------------- 합성 데이터

def _seed(*parts: str) -> int:
    return int(hashlib.md5("|".join(parts).encode()).hexdigest()[:8], 16)


def _cycle(t: np.ndarray, period: float = 36.0, phase: float = 0.0) -> np.ndarray:
    """반도체 업황을 흉내 낸 3년 주기 사인파 (-1~1)."""
    return np.sin(2 * np.pi * t / period + phase)


class SampleDataProvider(DataProvider):
    """API 키 없이 돌아가는 결정적 합성 데이터. 시연 백업과 테스트용.

    값은 실제 시세·재무가 아닙니다. 출처 type이 SAMPLE로 표시됩니다.
    """

    def __init__(self, base_date: date = date(2026, 10, 23)):
        self.base_date = base_date

    def _bdays(self, end: date, days: int) -> pd.DatetimeIndex:
        return pd.bdate_range(end=pd.Timestamp(end), periods=days)

    def get_ohlcv(self, ticker, end, days):
        idx = self._bdays(end, days)
        rng = np.random.default_rng(_seed("px", ticker))
        n = len(idx)
        t = np.arange(n) / 21.0  # 월 단위
        beta = 0.6 + rng.random() * 0.8
        drift = 0.0004 + 0.0012 * beta * np.cos(2 * np.pi * t / 36.0)  # 업황 따라 추세
        vol = 0.018 + 0.012 * rng.random()
        rets = drift + vol * rng.standard_normal(n)
        start_px = float(rng.integers(20, 300) * 1000)
        close = start_px * np.exp(np.cumsum(rets))
        open_ = close * (1 + 0.004 * rng.standard_normal(n))
        high = np.maximum(open_, close) * (1 + np.abs(0.008 * rng.standard_normal(n)))
        low = np.minimum(open_, close) * (1 - np.abs(0.008 * rng.standard_normal(n)))
        base_vol = rng.integers(200_000, 3_000_000)
        volume = (base_vol * np.exp(0.4 * rng.standard_normal(n)) * (1 + 3 * np.abs(rets))).astype(int)
        df = pd.DataFrame(
            {"open": open_, "high": high, "low": low, "close": close, "volume": volume}, index=idx
        )
        return df.round({"open": 0, "high": 0, "low": 0, "close": 0})

    def get_index(self, code, end, days):
        idx = self._bdays(end, days)
        rng = np.random.default_rng(_seed("idx", code))
        rets = 0.0003 + 0.011 * rng.standard_normal(len(idx))
        return pd.Series(2500 * np.exp(np.cumsum(rets)), index=idx, name=code)

    def get_flows(self, ticker, end, days):
        px = self.get_ohlcv(ticker, end, days)
        rng = np.random.default_rng(_seed("flow", ticker))
        value = px["close"] * px["volume"]
        trend = px["close"].pct_change(20).fillna(0).clip(-0.3, 0.3)
        foreign = value * (0.15 * trend + 0.05 * rng.standard_normal(len(px)))
        inst = value * (0.08 * trend + 0.04 * rng.standard_normal(len(px)))
        return pd.DataFrame(
            {"foreign": foreign, "institution": inst, "individual": -(foreign + inst)}, index=px.index
        ).round(0)

    def get_financials(self, ticker):
        rng = np.random.default_rng(_seed("fin", ticker))
        quarters = pd.period_range(end=pd.Period(self.base_date, "Q") - 1, periods=12, freq="Q")
        t = np.arange(12) * 3.0 + 9.0  # 수출 데이터와 같은 시간축(월)
        sens = 0.3 + 0.6 * rng.random()  # 업황 민감도
        base = float(rng.integers(500, 20000))
        cyc = _cycle(t, phase=1.2)
        revenue = base * (1 + sens * cyc + 0.03 * np.arange(12) / 4) * (1 + 0.03 * rng.standard_normal(12))
        margin = np.clip(0.08 + 0.12 * sens * cyc + 0.03 * rng.standard_normal(12), -0.15, 0.45)
        op = revenue * margin
        inventory = revenue * (0.5 - 0.25 * cyc) * (0.8 + 0.4 * rng.random())
        equity = base * 4 * (1 + 0.02 * np.arange(12))
        liabilities = equity * (0.3 + 0.9 * rng.random())
        return pd.DataFrame(
            {
                "quarter": [str(q) for q in quarters],
                "revenue": revenue.round(0),
                "operating_income": op.round(0),
                "net_income": (op * 0.78).round(0),
                "inventory": inventory.round(0),
                "capex": (revenue * (0.05 + 0.25 * rng.random())).round(0),
                "total_liabilities": liabilities.round(0),
                "total_equity": equity.round(0),
            }
        )

    def get_exports(self, end, months):
        idx = pd.period_range(end=pd.Period(end, "M") - 1, periods=months, freq="M")
        rng = np.random.default_rng(_seed("exports"))
        t = np.arange(len(idx)) + (60 - months)
        value = 9000 + 2600 * _cycle(t.astype(float), phase=1.2) + 25 * t + 250 * rng.standard_normal(len(idx))
        return pd.DataFrame({"value": value.round(0)}, index=[str(p) for p in idx])

    def get_profile(self, ticker):
        rng = np.random.default_rng(_seed("profile", ticker))
        info = UNIVERSE.get(ticker, {"name": ticker, "segment": "기타", "detail": ""})
        return {
            "name": info["name"],
            "segment": info["segment"],
            "products": info["detail"],
            "customers": "국내외 메모리·파운드리 업체 (샘플)",
            "top_customer_share": round(float(0.15 + 0.6 * rng.random()), 2),
            "moat": "공정 인증 이력과 고객사 공동 개발 (샘플)",
        }

    def search_filings(self, ticker, query, k=3):
        name = name_of(ticker)
        seg = segment_of(ticker)
        docs = [
            {
                "text": f"{name}은(는) {seg} 분야에서 {UNIVERSE.get(ticker, {}).get('detail', '')} 사업을 영위합니다. (샘플 공시 문장)",
                "title": f"{name} 사업보고서 - 사업의 내용 (샘플)",
                "date": "2026-03-15",
                "url": None,
            },
            {
                "text": f"{name}의 매출은 반도체 업황과 주요 고객사의 설비투자 계획에 영향을 받습니다. (샘플 공시 문장)",
                "title": f"{name} 사업보고서 - 위험 요인 (샘플)",
                "date": "2026-03-15",
                "url": None,
            },
        ]
        return docs[:k]

    def source(self, kind, ticker=None):
        label = {
            "price": "일별 시세",
            "index": "KOSPI 지수",
            "flow": "투자자별 순매수",
            "financials": "분기 재무제표",
            "exports": "반도체 월별 수출액",
            "profile": "기업 개요",
            "filing": "사업보고서 원문",
        }[kind]
        who = f"{name_of(ticker)} " if ticker else ""
        return Source(
            id=f"src_{kind}_{ticker}" if ticker else f"src_{kind}",
            type="SAMPLE",
            title=f"{who}{label} (합성 샘플 데이터)",
            date=str(self.base_date),
        )


# ---------------------------------------------------------------- 실데이터

class PykrxProvider(SampleDataProvider):
    """시세·지수·수급은 pykrx(KRX 공개 데이터), 나머지는 상위 클래스(샘플)에서.

    재무·수출·공시는 데이터 담당이 NEON에 적재한 뒤 PostgresProvider를 쓰세요.
    """

    def __init__(self, base_date: date | None = None):
        super().__init__(base_date or date.today())
        from pykrx import stock  # 지연 import: 설치 안 해도 패키지는 동작

        self._stock = stock

    @staticmethod
    def _range(end: date, days: int) -> tuple[str, str]:
        start = end - timedelta(days=int(days * 1.5) + 10)
        return start.strftime("%Y%m%d"), end.strftime("%Y%m%d")

    def get_ohlcv(self, ticker, end, days):
        s, e = self._range(end, days)
        df = self._stock.get_market_ohlcv(s, e, ticker)
        df = df.rename(columns={"시가": "open", "고가": "high", "저가": "low", "종가": "close", "거래량": "volume"})
        return df[["open", "high", "low", "close", "volume"]].tail(days)

    def get_index(self, code, end, days):
        s, e = self._range(end, days)
        return self._stock.get_index_ohlcv(s, e, code)["종가"].tail(days)

    def get_flows(self, ticker, end, days):
        s, e = self._range(end, days)
        df = self._stock.get_market_trading_value_by_date(s, e, ticker)
        out = pd.DataFrame(
            {"foreign": df["외국인합계"], "institution": df["기관합계"], "individual": df["개인"]}
        )
        return out.tail(days)

    def source(self, kind, ticker=None):
        if kind in ("price", "index", "flow"):
            src = super().source(kind, ticker)
            return Source(
                id=src.id,
                type="KRX",
                title=src.title.replace("(합성 샘플 데이터)", "(KRX, pykrx)"),
                date=str(self.base_date),
                url="https://data.krx.co.kr",
            )
        return super().source(kind, ticker)


class PostgresProvider(DataProvider):
    """NEON(PostgreSQL)에서 읽는다. 테이블 규격은 sql/schema.sql 참고.

    데이터 담당과 테이블·컬럼 이름을 맞추는 것이 이 클래스의 전부다.
    """

    def __init__(self, dsn: str, base_date: date | None = None):
        import psycopg

        self._conn = psycopg.connect(dsn)
        self.base_date = base_date or date.today()

    def _df(self, sql: str, params: tuple) -> pd.DataFrame:
        with self._conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [c.name for c in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=cols)

    def get_ohlcv(self, ticker, end, days):
        df = self._df(
            "SELECT trade_date, open, high, low, close, volume FROM daily_price "
            "WHERE ticker=%s AND trade_date<=%s ORDER BY trade_date DESC LIMIT %s",
            (ticker, end, days),
        )
        return df.set_index(pd.to_datetime(df.pop("trade_date"))).sort_index().astype(float)

    def get_index(self, code, end, days):
        df = self._df(
            "SELECT trade_date, close FROM daily_index WHERE index_code=%s AND trade_date<=%s "
            "ORDER BY trade_date DESC LIMIT %s",
            (code, end, days),
        )
        return df.set_index(pd.to_datetime(df["trade_date"]))["close"].sort_index().astype(float)

    def get_flows(self, ticker, end, days):
        df = self._df(
            "SELECT trade_date, foreign_net AS foreign, institution_net AS institution, "
            "individual_net AS individual FROM daily_flow WHERE ticker=%s AND trade_date<=%s "
            "ORDER BY trade_date DESC LIMIT %s",
            (ticker, end, days),
        )
        return df.set_index(pd.to_datetime(df.pop("trade_date"))).sort_index().astype(float)

    def get_financials(self, ticker):
        df = self._df(
            "SELECT quarter, revenue, operating_income, net_income, inventory, capex, "
            "total_liabilities, total_equity FROM quarterly_financials WHERE ticker=%s "
            "ORDER BY quarter DESC LIMIT 12",
            (ticker,),
        )
        return df.iloc[::-1].reset_index(drop=True)

    def get_exports(self, end, months):
        df = self._df(
            "SELECT to_char(month, 'YYYY-MM') AS month, value FROM semiconductor_exports "
            "WHERE month<=%s ORDER BY month DESC LIMIT %s",
            (end, months),
        )
        return df.set_index("month").sort_index().astype(float)

    def get_profile(self, ticker):
        df = self._df("SELECT * FROM company_profile WHERE ticker=%s", (ticker,))
        if df.empty:
            info = UNIVERSE.get(ticker, {})
            return {"name": info.get("name", ticker), "segment": info.get("segment", "기타"),
                    "products": info.get("detail", ""), "customers": None,
                    "top_customer_share": None, "moat": None}
        return df.iloc[0].to_dict()

    def search_filings(self, ticker, query, k=3):
        # pgvector 임베딩 검색은 RAG 담당 모듈로 교체. 우선 키워드 검색.
        df = self._df(
            "SELECT chunk_text AS text, report_title AS title, filed_date::text AS date, url "
            "FROM filing_chunks WHERE ticker=%s AND chunk_text ILIKE %s LIMIT %s",
            (ticker, f"%{query}%", k),
        )
        return df.to_dict("records")

    def source(self, kind, ticker=None):
        mapping = {
            "price": ("KIS", "일별 시세"), "index": ("KRX", "지수"), "flow": ("KIS", "투자자별 순매수"),
            "financials": ("DART", "분기 재무제표"), "exports": ("PUBLIC_DATA", "반도체 월별 수출액"),
            "profile": ("DART", "기업 개요"), "filing": ("DART", "사업보고서 원문"),
        }
        typ, label = mapping[kind]
        who = f"{name_of(ticker)} " if ticker else ""
        return Source(
            id=f"src_{kind}_{ticker}" if ticker else f"src_{kind}",
            type=typ, title=f"{who}{label}", date=str(self.base_date),
        )

### `kic_agents/tools/technical.py`
기술적 분석 도구: 지표, 추세 국면(D), 패턴·적중률(E), 지지·저항, 수급, 위험

In [ ]:
%%writefile kic_agents/tools/technical.py
"""기술적 분석 도구 (설계서 6-2). 숫자는 전부 여기서 계산하고 LLM은 계산하지 않는다.

분류 과제 D(추세 국면)와 E(차트 패턴, 규칙 기반)를 포함한다.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

FORWARD_DAYS = 20  # 신호 적중률을 볼 기간
MIN_SAMPLES = 10  # 이보다 적으면 "참고용"


# ---------------------------------------------------------------- 지표

def _rsi(close: pd.Series, n: int = 14) -> pd.Series:
    diff = close.diff()
    up = diff.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    down = (-diff.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    rs = up / down.replace(0, np.nan)
    return (100 - 100 / (1 + rs)).fillna(100)


def _atr(df: pd.DataFrame, n: int = 14) -> pd.Series:
    prev = df["close"].shift()
    tr = pd.concat([df["high"] - df["low"], (df["high"] - prev).abs(), (df["low"] - prev).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


def _adx(df: pd.DataFrame, n: int = 14) -> pd.Series:
    up = df["high"].diff()
    down = -df["low"].diff()
    plus_dm = np.where((up > down) & (up > 0), up, 0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)
    atr = _atr(df, n)
    plus_di = 100 * pd.Series(plus_dm, index=df.index).ewm(alpha=1 / n, adjust=False).mean() / atr
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(alpha=1 / n, adjust=False).mean() / atr
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    return dx.ewm(alpha=1 / n, adjust=False).mean().fillna(0)


def calc_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """OHLCV에 지표 열을 붙여 돌려준다."""
    out = df.copy()
    c = out["close"]
    for n in (5, 20, 60, 120):
        out[f"ma{n}"] = c.rolling(n).mean()
    out["rsi14"] = _rsi(c)
    ema12, ema26 = c.ewm(span=12, adjust=False).mean(), c.ewm(span=26, adjust=False).mean()
    out["macd"] = ema12 - ema26
    out["macd_signal"] = out["macd"].ewm(span=9, adjust=False).mean()
    std20 = c.rolling(20).std()
    out["bb_upper"], out["bb_lower"] = out["ma20"] + 2 * std20, out["ma20"] - 2 * std20
    out["atr14"] = _atr(out)
    out["atr_pct"] = out["atr14"] / c
    out["adx14"] = _adx(out)
    out["obv"] = (np.sign(c.diff().fillna(0)) * out["volume"]).cumsum()
    out["vol_ma20"] = out["volume"].rolling(20).mean()
    return out


# ---------------------------------------------------------------- 분류 과제 D

def classify_trend(ind: pd.DataFrame) -> dict:
    """추세 국면: 상승추세 / 하락추세 / 박스권 / 변동성 확대 + 근거 값."""
    last = ind.iloc[-1]
    ma60_slope = (ind["ma60"].iloc[-1] / ind["ma60"].iloc[-21] - 1) if len(ind) > 81 else 0.0
    atr_ratio = last["atr_pct"] / ind["atr_pct"].tail(120).median()
    if last["close"] > last["ma20"] > last["ma60"]:
        alignment = "정배열"
    elif last["close"] < last["ma20"] < last["ma60"]:
        alignment = "역배열"
    else:
        alignment = "혼조"

    if atr_ratio >= 1.5:
        phase = "변동성 확대"
    elif alignment == "정배열" and last["adx14"] >= 20 and ma60_slope > 0:
        phase = "상승추세"
    elif alignment == "역배열" and last["adx14"] >= 20 and ma60_slope < 0:
        phase = "하락추세"
    else:
        phase = "박스권"
    return {
        "phase": phase,
        "evidence": [
            {"name": "이동평균 배열", "value": alignment},
            {"name": "ADX", "value": round(float(last["adx14"]), 1)},
            {"name": "60일선 한 달 기울기(%)", "value": round(float(ma60_slope) * 100, 1)},
            {"name": "변동성 배수(평소 대비)", "value": round(float(atr_ratio), 2)},
        ],
    }


# ---------------------------------------------------------------- 분류 과제 E

def signal_series(ind: pd.DataFrame) -> dict[str, pd.Series]:
    """날짜별 신호 발생 여부. 패턴 탐지와 적중률 계산이 같은 정의를 쓴다."""
    ma20, ma60 = ind["ma20"], ind["ma60"]
    hi60 = ind["high"].rolling(60).max().shift()
    lo60 = ind["low"].rolling(60).min().shift()
    vol_spike = ind["volume"] > 1.5 * ind["vol_ma20"]
    rsi = ind["rsi14"]
    return {
        "골든크로스(20/60)": (ma20 > ma60) & (ma20.shift() <= ma60.shift()),
        "데드크로스(20/60)": (ma20 < ma60) & (ma20.shift() >= ma60.shift()),
        "60일 고점 돌파(거래량 동반)": (ind["close"] > hi60) & vol_spike,
        "60일 저점 이탈": ind["close"] < lo60,
        "RSI 과매수 진입(70)": (rsi > 70) & (rsi.shift() <= 70),
        "RSI 과매도 진입(30)": (rsi < 30) & (rsi.shift() >= 30),
        "이중바닥 확인": _double_bottom(ind),
    }


def _double_bottom(ind: pd.DataFrame, lookback: int = 60, tol: float = 0.03) -> pd.Series:
    """최근 60일 안의 두 저점이 3% 이내이고 10일 이상 떨어져 있으며, 오늘 종가가 그 사이 고점(넥라인)을 넘으면 True."""
    low, close, high = ind["low"].values, ind["close"].values, ind["high"].values
    res = np.zeros(len(ind), dtype=bool)
    for i in range(lookback, len(ind)):
        w = slice(i - lookback, i)
        lw = low[w]
        a = int(np.argmin(lw))
        mask = np.ones_like(lw, dtype=bool)
        mask[max(0, a - 10): a + 11] = False
        if not mask.any():
            continue
        b = int(np.argmin(np.where(mask, lw, np.inf)))
        if abs(lw[a] - lw[b]) / lw[a] > tol:
            continue
        lo_i, hi_i = sorted((a, b))
        neckline = high[w][lo_i: hi_i + 1].max()
        res[i] = close[i] > neckline and close[i - 1] <= neckline
    return pd.Series(res, index=ind.index)


def signal_hit_rate(ind: pd.DataFrame, series: pd.Series, bullish: bool = True) -> dict:
    """과거에 같은 신호가 났을 때 이후 20일 수익률 분포. bullish=False면 하락을 적중으로 본다."""
    fwd = ind["close"].shift(-FORWARD_DAYS) / ind["close"] - 1
    hits = fwd[series & fwd.notna()]
    n = int(len(hits))
    if n == 0:
        return {"n_samples": 0, "hit_rate_20d": None, "avg_return_20d": None, "reliability": "참고용"}
    win = (hits > 0) if bullish else (hits < 0)
    reliability = "참고용" if n < MIN_SAMPLES else ("높음" if n >= 30 else "보통")
    return {
        "n_samples": n,
        "hit_rate_20d": round(float(win.mean()) * 100, 1),
        "avg_return_20d": round(float(hits.mean()) * 100, 1),
        "reliability": reliability,
    }


_BEARISH = {"데드크로스(20/60)", "60일 저점 이탈", "RSI 과매수 진입(70)"}


def detect_patterns(ind: pd.DataFrame, window: int = 20) -> list[dict]:
    """최근 window일 안에 난 신호 + 과거 적중률."""
    out = []
    for name, s in signal_series(ind).items():
        recent = s.tail(window)
        if not recent.any():
            continue
        last_day = recent[recent].index[-1]
        stats = signal_hit_rate(ind, s, bullish=name not in _BEARISH)
        out.append({"type": name, "date": last_day.strftime("%Y-%m-%d"), "direction": "하락" if name in _BEARISH else "상승", **stats})
    return sorted(out, key=lambda x: x["date"], reverse=True)


# ---------------------------------------------------------------- 가격대·수급·위험

def support_resistance(ind: pd.DataFrame, lookback: int = 120, k: int = 2) -> dict:
    """최근 120일 스윙 고점·저점을 2% 단위로 묶어 현재가 아래(지지)·위(저항)에서 가까운 순으로."""
    w = ind.tail(lookback)
    price = float(w["close"].iloc[-1])
    lows = w["low"][(w["low"] == w["low"].rolling(11, center=True).min())].tolist()
    highs = w["high"][(w["high"] == w["high"].rolling(11, center=True).max())].tolist()

    def cluster(levels: list[float]) -> list[float]:
        levels = sorted(levels)
        groups: list[list[float]] = []
        for lv in levels:
            if groups and abs(lv - groups[-1][-1]) / groups[-1][-1] <= 0.02:
                groups[-1].append(lv)
            else:
                groups.append([lv])
        return [round(float(np.mean(g)), -1) for g in groups]

    levels = cluster(lows + highs)
    support = sorted([lv for lv in levels if lv < price], reverse=True)[:k]
    resistance = sorted([lv for lv in levels if lv > price])[:k]
    hist, edges = np.histogram(w["close"], bins=20, weights=w["volume"])
    top = int(np.argmax(hist))
    volume_node = round(float((edges[top] + edges[top + 1]) / 2), -1)
    return {"support": support, "resistance": resistance, "volume_node": [volume_node]}


def flow_summary(flows: pd.DataFrame, price: pd.DataFrame, days: int = 20) -> dict:
    """최근 20일 누적 순매수(억원)와 거래대금 대비 비율(%)."""
    f = flows.tail(days)
    value = float((price["close"] * price["volume"]).tail(days).sum()) or 1.0
    out = {f"{col}_{days}d_억원": round(float(f[col].sum()) / 1e8, 1) for col in ("foreign", "institution", "individual")}
    out["foreign_inst_ratio_pct"] = round(float(f["foreign"].sum() + f["institution"].sum()) / value * 100, 2)
    return out


def risk_metrics(ind: pd.DataFrame, index_close: pd.Series | None = None) -> dict:
    r = ind["close"].pct_change().dropna()
    last_year = ind["close"].tail(252)
    mdd = float((last_year / last_year.cummax() - 1).min())
    beta = None
    if index_close is not None:
        joined = pd.concat([r, index_close.pct_change()], axis=1, join="inner").dropna().tail(252)
        if len(joined) > 30:
            cov = np.cov(joined.iloc[:, 0], joined.iloc[:, 1])
            beta = round(float(cov[0, 1] / cov[1, 1]), 2)
    return {
        "volatility_annual_pct": round(float(r.tail(252).std() * np.sqrt(252)) * 100, 1),
        "daily_move_pct": round(float(r.tail(60).abs().mean()) * 100, 2),
        "mdd_1y_pct": round(mdd * 100, 1),
        "beta": beta,
    }


def returns(ind: pd.DataFrame) -> dict:
    c = ind["close"]
    return {f"return_{n}d_pct": round(float(c.iloc[-1] / c.iloc[-1 - n] - 1) * 100, 1) for n in (20, 60, 120) if len(c) > n}


# ---------------------------------------------------------------- 팩터 점수 (0~1)

def factor_scores(ind: pd.DataFrame, flow: dict) -> dict:
    ret60 = float(ind["close"].iloc[-1] / ind["close"].iloc[-61] - 1)
    trend_bonus = 0.1 if ind["ma20"].iloc[-1] > ind["ma60"].iloc[-1] else -0.1
    momentum = 0.5 + 0.5 * np.tanh(ret60 / 0.25) + trend_bonus
    flow_score = 0.5 + 0.5 * np.tanh(flow["foreign_inst_ratio_pct"] / 5)
    return {"momentum": round(float(np.clip(momentum, 0, 1)), 3), "flow": round(float(flow_score), 3)}

### `kic_agents/tools/industry.py`
산업·사업모델 도구: 재무 파생지표, 업황 국면(C), 동종사 비교

In [ ]:
%%writefile kic_agents/tools/industry.py
"""산업·사업모델 분석 도구 (설계서 5-2). 분류 과제 C(업황 국면)를 포함한다."""

from __future__ import annotations

import numpy as np
import pandas as pd


def derive_financials(fin: pd.DataFrame) -> dict:
    """최근 분기 기준 파생지표. 비율은 % 단위."""
    f = fin.reset_index(drop=True)
    last = f.iloc[-1]
    yoy_base = f.iloc[-5] if len(f) >= 5 else None
    ttm_rev = float(f["revenue"].tail(4).sum())
    avg_inv = float(f["inventory"].tail(2).mean())

    def pct(a, b):
        return round(float(a / b - 1) * 100, 1) if b not in (0, None) else None

    return {
        "quarter": last["quarter"],
        "revenue_억원": float(last["revenue"]),
        "revenue_yoy_pct": pct(last["revenue"], yoy_base["revenue"]) if yoy_base is not None else None,
        "op_margin_pct": round(float(last["operating_income"] / last["revenue"]) * 100, 1),
        "op_margin_change_pp": round(
            float(last["operating_income"] / last["revenue"] - f.iloc[-2]["operating_income"] / f.iloc[-2]["revenue"]) * 100, 1
        ) if len(f) >= 2 else None,
        "inventory_turnover": round(ttm_rev / avg_inv, 2) if avg_inv else None,
        "inventory_turnover_prev": round(float(f["revenue"].iloc[-8:-4].sum()) / float(f["inventory"].iloc[-6:-4].mean()), 2)
        if len(f) >= 8 else None,
        "debt_ratio_pct": round(float(last["total_liabilities"] / last["total_equity"]) * 100, 1),
        "capex_to_revenue_pct": round(float(f["capex"].tail(4).sum()) / ttm_rev * 100, 1),
    }


def export_trend(exports: pd.DataFrame) -> pd.DataFrame:
    e = exports.copy()
    e["yoy_pct"] = (e["value"] / e["value"].shift(12) - 1) * 100
    e["yoy_3m_change_pp"] = e["yoy_pct"] - e["yoy_pct"].shift(3)
    return e


def classify_cycle(exports: pd.DataFrame, derived: dict | None = None) -> dict:
    """업황 국면: 회복 / 확장 / 둔화 / 수축.

    수출 YoY 부호(+/-)와 최근 3개월 방향(상승/하락)의 조합.
    재고회전율 개선 여부는 근거로 함께 보여 준다.
    """
    e = export_trend(exports).dropna()
    last = e.iloc[-1]
    yoy, slope = float(last["yoy_pct"]), float(last["yoy_3m_change_pp"])
    if yoy >= 0:
        phase = "확장" if slope >= 0 else "둔화"
    else:
        phase = "회복" if slope >= 0 else "수축"
    rising_months = 0
    for v in reversed(e["yoy_pct"].diff().dropna().tolist()):
        if (v >= 0) == (slope >= 0):
            rising_months += 1
        else:
            break
    evidence = [
        {"name": "반도체 수출 YoY(%)", "value": round(yoy, 1), "source": "exports"},
        {"name": "수출 YoY 3개월 변화(%p)", "value": round(slope, 1), "source": "exports"},
        {"name": "같은 방향 지속(개월)", "value": rising_months, "source": "exports"},
        {"name": "기준월", "value": str(e.index[-1]), "source": "exports"},
    ]
    if derived and derived.get("inventory_turnover") and derived.get("inventory_turnover_prev"):
        better = derived["inventory_turnover"] > derived["inventory_turnover_prev"]
        evidence.append({"name": "재고회전율(회, 전년 대비)", "value": f"{derived['inventory_turnover_prev']} → {derived['inventory_turnover']} ({'개선' if better else '악화'})", "source": "financials"})
    return {"phase": phase, "evidence": evidence}


def percentile(values: dict[str, float], key: str) -> float | None:
    """values 안에서 key 값의 백분위(0~1). 값이 하나면 0.5."""
    clean = {k: v for k, v in values.items() if v is not None and not np.isnan(v)}
    if key not in clean:
        return None
    if len(clean) == 1:
        return 0.5
    arr = np.array(list(clean.values()))
    return float((arr < clean[key]).sum() + 0.5 * ((arr == clean[key]).sum() - 1)) / (len(arr) - 1)


def compare_peers(ticker: str, derived_all: dict[str, dict], segments: dict[str, str], names: dict[str, str]) -> list[dict]:
    """같은 밸류체인 안 동종사(3개 미만이면 전체 유니버스)의 핵심 지표 표."""
    seg = segments.get(ticker)
    peers = [t for t in derived_all if segments.get(t) == seg]
    if len(peers) < 3:
        peers = list(derived_all)
    rows = []
    for t in peers:
        d = derived_all[t]
        rows.append({
            "ticker": t, "name": names.get(t, t),
            "revenue_yoy_pct": d["revenue_yoy_pct"], "op_margin_pct": d["op_margin_pct"],
            "debt_ratio_pct": d["debt_ratio_pct"], "is_target": t == ticker,
        })
    return sorted(rows, key=lambda r: (r["op_margin_pct"] is None, -(r["op_margin_pct"] or 0)))


def factor_scores(ticker: str, derived_all: dict[str, dict]) -> dict:
    """유니버스 안 백분위로 성장·수익성·안정성 점수 (0~1)."""
    g = percentile({t: d["revenue_yoy_pct"] for t, d in derived_all.items()}, ticker)
    p = percentile({t: d["op_margin_pct"] for t, d in derived_all.items()}, ticker)
    s = percentile({t: d["debt_ratio_pct"] for t, d in derived_all.items()}, ticker)
    out = {}
    if g is not None:
        out["growth"] = round(g, 3)
    if p is not None:
        out["profitability"] = round(p, 3)
    if s is not None:
        out["stability"] = round(1 - s, 3)
    return out


def rule_items(derived: dict, cycle: dict, profile: dict, src: dict[str, str]) -> tuple[list[dict], list[dict]]:
    """데이터에서 바로 뽑는 기회·리스크 후보. LLM이 없어도 각각 최소 1개를 보장한다."""
    opp, risk = [], []
    phase = cycle["phase"]
    if phase in ("회복", "확장"):
        opp.append({"text": f"반도체 수출이 {phase} 국면이라 업황이 실적을 밀어 줄 수 있음", "source_id": src["exports"]})
    else:
        risk.append({"text": f"반도체 수출이 {phase} 국면이라 업황이 실적에 부담이 될 수 있음", "source_id": src["exports"]})
    yoy = derived.get("revenue_yoy_pct")
    if yoy is not None and yoy >= 10:
        opp.append({"text": f"매출이 전년 같은 분기보다 {yoy}% 늘어남", "source_id": src["financials"]})
    elif yoy is not None and yoy < 0:
        risk.append({"text": f"매출이 전년 같은 분기보다 {abs(yoy)}% 줄어듦", "source_id": src["financials"]})
    chg = derived.get("op_margin_change_pp")
    if chg is not None and chg >= 2:
        opp.append({"text": f"영업이익률이 직전 분기보다 {chg}%p 개선됨", "source_id": src["financials"]})
    elif chg is not None and chg <= -2:
        risk.append({"text": f"영업이익률이 직전 분기보다 {abs(chg)}%p 낮아짐", "source_id": src["financials"]})
    if derived.get("debt_ratio_pct", 0) >= 100:
        risk.append({"text": f"부채비율이 {derived['debt_ratio_pct']}%로 자기자본보다 빚이 많음", "source_id": src["financials"]})
    if derived.get("capex_to_revenue_pct", 0) >= 20:
        risk.append({"text": f"매출 대비 설비투자가 {derived['capex_to_revenue_pct']}%로 커서 업황이 꺾이면 부담", "source_id": src["financials"]})
    share = profile.get("top_customer_share")
    if share is not None and share >= 0.4:
        risk.append({"text": f"최대 고객 매출 비중이 {round(share * 100)}%로 고객 집중도가 높음", "source_id": src["profile"]})
    if not opp:
        opp.append({"text": f"재무 안정성 지표(부채비율 {derived['debt_ratio_pct']}%)는 업황 반등 시 버틸 여력이 됨", "source_id": src["financials"]})
    if not risk:
        risk.append({"text": "반도체는 업황 주기가 뚜렷해, 지금 좋은 실적이 다음 사이클에도 이어진다는 보장은 없음", "source_id": src["exports"]})
    return opp, risk

### `kic_agents/llm.py`
Claude 호출 + 숫자·출처 자체 점검

In [ ]:
%%writefile kic_agents/llm.py
"""문장 작성 단계. 도구가 계산한 facts만 받아 페르소나에 맞는 요약을 쓴다.

- Claude API(structured output)로 요약·기회·리스크·용어 풀이를 JSON으로 받는다.
- 받은 문장의 숫자가 facts에 없는 숫자면 한 번 다시 쓰게 하고, 그래도 틀리면 템플릿으로 대체한다.
- ANTHROPIC_API_KEY가 없거나 KIC_USE_LLM=0이면 LLM을 부르지 않는다.
"""

from __future__ import annotations

import json
import logging
import os
import re
from typing import Any

from pydantic import BaseModel

from kic_agents.personas import Persona

log = logging.getLogger(__name__)

MODEL = os.getenv("KIC_MODEL", "claude-opus-5-5")


class LLMItem(BaseModel):
    text: str
    source_id: str


class LLMTerm(BaseModel):
    term: str
    explanation: str


class LLMOutput(BaseModel):
    summary: str
    opportunities: list[LLMItem]
    risks: list[LLMItem]
    terms: list[LLMTerm]


EMPHASIS_TEXT = {
    "opportunity_first": "기회 요인을 먼저 쓰되, 리스크를 반드시 1개 이상 함께 쓴다 (과신 방지).",
    "risk_first": "리스크 요인을 먼저 쓰되, 놓치는 기회도 반드시 1개 이상 함께 쓴다 (과도한 공포 방지).",
}
TERMS_TEXT = {
    "all": "전문 용어는 모두 쉬운 말로 풀고, 비유를 쓴다.",
    "first": "처음 나오는 전문 용어 2개까지만 풀이한다.",
    "none": "용어 풀이는 하지 않는다.",
}


def system_prompt(role: str, goal: str, persona: Persona) -> str:
    lo, hi = persona.summary_len
    length = f"{lo}문장" if lo == hi else f"{lo}~{hi}문장"
    return f"""당신은 {role}입니다.
목표: {goal}

규칙
1. 숫자는 사용자 메시지의 facts에 있는 값만 쓴다. 새로 계산하거나 추정하지 않는다.
2. opportunities와 risks는 각각 1개 이상이고, source_id는 facts.sources에 있는 id만 쓴다.
3. 매수·매도 추천, 목표주가, "오른다/내린다" 같은 예측은 말하지 않는다. 현재 상태와 근거만 말한다.
4. 자료가 없으면 "확인 불가"라고 쓴다.
5. 한국어 존댓말(해요체)로 쓴다.

사용자 페르소나: {persona.id} {persona.label}
- summary는 {length}
- 지표는 최대 {persona.max_metrics}개만 언급
- {TERMS_TEXT[persona.explain_terms]}
- {EMPHASIS_TEXT[persona.emphasis]}
- 관점 기간: {"단기(1~3개월)" if persona.horizon == "short" else "장기(1년 이상)"}"""


# ---------------------------------------------------------------- 숫자 검증 (자체 점검)

_NUM = re.compile(r"(?<![\w.])-?\d[\d,]*(?:\.\d+)?")
_UNIT_AFTER = re.compile(r"^\s*(일|개월|년|주|개|번|분기|문장|위|등|차|월|배 이상)")


def _collect_numbers(obj: Any, out: list[float]) -> None:
    if isinstance(obj, bool):
        return
    if isinstance(obj, (int, float)):
        out.append(float(obj))
    elif isinstance(obj, str):
        out.extend(float(m.replace(",", "")) for m in _NUM.findall(obj))
    elif isinstance(obj, dict):
        for v in obj.values():
            _collect_numbers(v, out)
    elif isinstance(obj, (list, tuple)):
        for v in obj:
            _collect_numbers(v, out)


def unknown_numbers(text: str, facts: dict) -> list[str]:
    """text 안의 숫자 중 facts에서 찾을 수 없는 것. 기간·개수 같은 단위 숫자는 건너뛴다."""
    known: list[float] = []
    _collect_numbers(facts, known)
    bad = []
    for m in _NUM.finditer(text):
        tail = text[m.end(): m.end() + 4]
        if _UNIT_AFTER.match(tail):
            continue
        n = float(m.group().replace(",", ""))
        if not any(abs(n - k) <= max(0.051, abs(k) * 0.01) or abs(abs(n) - abs(k)) <= 0.051 for k in known):
            bad.append(m.group())
    return bad


# ---------------------------------------------------------------- Claude 호출

def llm_enabled() -> bool:
    return bool(os.getenv("ANTHROPIC_API_KEY")) and os.getenv("KIC_USE_LLM", "1") != "0"


class Writer:
    def __init__(self, client=None):
        self._client = client

    @property
    def client(self):
        if self._client is None:
            import anthropic

            self._client = anthropic.Anthropic()
        return self._client

    def write(self, system: str, facts: dict, question: str = "") -> LLMOutput | None:
        """성공하면 검증된 LLMOutput, 실패하면 None (호출 측이 템플릿 사용)."""
        if self._client is None and not llm_enabled():
            return None
        import anthropic

        user = "facts:\n" + json.dumps(facts, ensure_ascii=False, default=str)
        if question:
            user += f"\n\n사용자 질문: {question}"
        messages: list[dict] = [{"role": "user", "content": user}]
        source_ids = {s["id"] for s in facts.get("sources", [])}

        for attempt in range(2):
            try:
                resp = self.client.beta.messages.parse(
                    model=MODEL,
                    max_tokens=4000,
                    system=system,
                    messages=messages,
                    output_format=LLMOutput,
                    output_config={"effort": "low"},
                    betas=["server-side-fallback-2026-07-01"],
                    fallbacks="default",
                )
            except anthropic.RateLimitError as e:
                log.warning("Claude 속도 제한: %s", e)
                return None
            except anthropic.APIStatusError as e:
                log.warning("Claude API 오류 %s: %s", e.status_code, e.message)
                return None
            except anthropic.APIConnectionError as e:
                log.warning("Claude 연결 실패: %s", e)
                return None

            if resp.stop_reason == "refusal" or resp.parsed_output is None:
                log.warning("Claude가 응답을 만들지 못함: %s", resp.stop_reason)
                return None
            out: LLMOutput = resp.parsed_output

            problems = []
            all_text = " ".join([out.summary] + [i.text for i in out.opportunities + out.risks])
            bad_nums = unknown_numbers(all_text, facts)
            if bad_nums:
                problems.append(f"facts에 없는 숫자: {', '.join(bad_nums)}")
            bad_src = [i.source_id for i in out.opportunities + out.risks if i.source_id not in source_ids]
            if bad_src:
                problems.append(f"없는 source_id: {', '.join(bad_src)}")
            if not out.opportunities or not out.risks:
                problems.append("기회와 리스크는 각각 1개 이상이어야 함")
            if not problems:
                return out
            log.info("자체 점검 실패(%d회차): %s", attempt + 1, problems)
            messages += [
                # 응답 블록을 그대로 돌려준다 (thinking 블록 포함, 기록 수정 금지)
                {"role": "assistant", "content": resp.content},
                {"role": "user", "content": "다음 문제를 고쳐서 다시 써 주세요: " + "; ".join(problems)},
            ]
        return None

### `kic_agents/agents/base.py`
두 에이전트 공통 마무리 (지표 선택, 템플릿 요약)

In [ ]:
%%writefile kic_agents/agents/base.py
"""두 에이전트가 함께 쓰는 마무리 단계: 지표 고르기, 템플릿 요약, LLM 문장 적용."""

from __future__ import annotations

import re
from typing import Any

from kic_agents.llm import Writer
from kic_agents.personas import Persona, term_notes
from kic_agents.schemas import BaseResult, Evidence, Item, Source, TermNote

Metric = tuple[Any, str | None]  # (값, source_id)


def pick_metrics(available: dict[str, Metric], priority: list[str], persona: Persona) -> list[Evidence]:
    """페르소나 우선순위대로 max_metrics개. L4는 전부."""
    names = list(available) if persona.level == "L4" else [n for n in priority if n in available]
    names = names[: persona.max_metrics]
    return [Evidence(name=n, value=available[n][0], source_id=available[n][1]) for n in names]


def _fmt(v: Any) -> str:
    if isinstance(v, float):
        return f"{v:,.1f}" if abs(v) < 1000 else f"{v:,.0f}"
    if isinstance(v, list):
        return ", ".join(_fmt(x) for x in v) if v else "없음"
    return str(v)


def josa(word: str, pair: tuple[str, str] = ("은", "는")) -> str:
    """받침에 맞는 조사를 붙인다. 한글이 아니면 '은(는)'처럼 둘 다 쓴다."""
    ch = word[-1] if word else ""
    if "가" <= ch <= "힣":
        return word + (pair[0] if (ord(ch) - 0xAC00) % 28 else pair[1])
    return f"{word}{pair[0]}({pair[1]})"


def _metric_text(m: Evidence) -> str:
    unit = re.search(r"\(([^()]*)\)$", m.name)
    parts = [x.strip() for x in unit.group(1).split(",")] if unit else []
    if parts and parts[-1] in ("%", "%p", "원", "억원", "회"):
        label = m.name[: unit.start()].strip() + (f"({', '.join(parts[:-1])})" if parts[:-1] else "")
        return f"{label} {_fmt(m.value)}{parts[-1]}"
    return f"{m.name} {_fmt(m.value)}"


def template_summary(lead: str, metrics: list[Evidence], opp: list[dict], risk: list[dict], persona: Persona, closing: str) -> str:
    """LLM 없이 만드는 요약. 숫자는 metrics와 items에서만 가져오므로 환각이 없다."""
    sentences = [lead]
    if metrics and persona.level != "L1":
        sentences.append("주요 지표: " + ", ".join(_metric_text(m) for m in metrics[:4]) + ".")
    first, second = (opp, risk) if persona.emphasis == "opportunity_first" else (risk, opp)
    first_label, second_label = ("기회", "리스크") if persona.emphasis == "opportunity_first" else ("리스크", "기회")
    if first:
        sentences.append(f"{first_label} 요인: {first[0]['text']}.")
    if second:
        sentences.append(f"{second_label} 요인: {second[0]['text']}.")
    sentences.append(closing)
    _, hi = persona.summary_len
    return " ".join(sentences[: max(hi, 3)])


def finalize(
    result: BaseResult,
    *,
    persona: Persona,
    writer: Writer | None,
    system: str,
    facts: dict,
    question: str,
    opp: list[dict],
    risk: list[dict],
    lead: str,
    closing: str,
    glossary_terms: list[str],
) -> BaseResult:
    out = writer.write(system, facts, question) if writer is not None else None
    if out is not None:
        result.summary = out.summary
        result.opportunities = [Item(**i.model_dump()) for i in out.opportunities]
        result.risks = [Item(**i.model_dump()) for i in out.risks]
        result.terms = [TermNote(**t.model_dump()) for t in out.terms] if persona.explain_terms != "none" else []
        result.summary_by = "llm"
    else:
        result.summary = template_summary(lead, result.metrics, opp, risk, persona, closing)
        result.opportunities = [Item(**i) for i in opp]
        result.risks = [Item(**i) for i in risk]
        result.terms = [TermNote(**t) for t in term_notes(persona, glossary_terms)]
        result.summary_by = "template"
    return result


def facts_payload(result: BaseResult, extra: dict, opp: list[dict], risk: list[dict], sources: list[Source]) -> dict:
    return {
        "ticker": result.ticker,
        "name": result.name,
        "as_of": result.as_of,
        "metrics_to_mention": [m.model_dump() for m in result.metrics],
        **extra,
        "candidate_opportunities": opp,
        "candidate_risks": risk,
        "sources": [s.model_dump() for s in sources],
    }

### `kic_agents/agents/technical_agent.py`
기술적 분석 에이전트

In [ ]:
%%writefile kic_agents/agents/technical_agent.py
"""기술적 분석 에이전트 (설계서 6장).

흐름: 시세·수급 조회 → 지표 계산 → 추세 국면(D) → 최근 신호(E) + 과거 적중률
→ 지지·저항, 위험 지표 → 페르소나에 맞춰 지표 선택 → 문장 작성 → 자체 점검
"""

from __future__ import annotations

from kic_agents.agents.base import facts_payload, finalize, josa, pick_metrics
from kic_agents.data.providers import DataProvider
from kic_agents.data.universe import name_of
from kic_agents.llm import Writer, system_prompt
from kic_agents.personas import get_persona
from kic_agents.schemas import AgentRequest, Phase, Signal, TechnicalResult
from kic_agents.tools import technical as T

LOOKBACK_DAYS = 800  # 약 3년: 신호 적중률 표본용

# 설계서 6-5 표: 페르소나별로 보여줄 지표 순서
PRIORITY = {
    "L1-AGG": ["추세 국면", "최근 1년 최대낙폭(%)"],
    "L1-CON": ["추세 국면", "하루 평균 움직임(%)"],
    "L2-AGG": ["이동평균 배열", "RSI(14)", "거래량(20일 평균 대비 배)", "최근 신호"],
    "L2-CON": ["60일선 대비(%)", "120일선 대비(%)", "지지선(원)", "연환산 변동성(%)"],
    "L3-AGG": ["추세 국면", "MACD-시그널", "ADX", "최근 신호", "외국인 20일 순매수(억원)",
               "기관 20일 순매수(억원)", "60일 수익률(%)", "거래량(20일 평균 대비 배)"],
    "L3-CON": ["지지선(원)", "저항선(원)", "ATR(14, 원)", "최근 1년 최대낙폭(%)", "베타",
               "연환산 변동성(%)", "120일선 대비(%)", "매물대(원)"],
}
TERMS = {
    "L1": ["이동평균선", "최대낙폭", "변동성"],
    "L2": ["이동평균선", "RSI", "지지선", "변동성"],
}
PHASE_PLAIN = {"상승추세": "오르는", "하락추세": "내리는", "박스권": "일정한 범위 안에서 오르내리는", "변동성 확대": "평소보다 크게 흔들리는"}


def run(req: AgentRequest, provider: DataProvider, writer: Writer | None = None, use_llm: bool = True) -> TechnicalResult:
    persona = get_persona(req.persona_id)
    ticker = req.ticker
    end = req.as_of or provider.base_date

    px = provider.get_ohlcv(ticker, end, LOOKBACK_DAYS)
    index_close = provider.get_index("1001", end, LOOKBACK_DAYS)
    flows = provider.get_flows(ticker, end, 60)
    sources = [provider.source("price", ticker), provider.source("flow", ticker), provider.source("index")]
    sid = {"price": sources[0].id, "flow": sources[1].id, "index": sources[2].id}

    ind = T.calc_indicators(px)
    trend = T.classify_trend(ind)
    patterns = T.detect_patterns(ind)
    levels = T.support_resistance(ind)
    flow = T.flow_summary(flows, px)
    risk = T.risk_metrics(ind, index_close)
    rets = T.returns(ind)
    last = ind.iloc[-1]
    price = float(last["close"])

    available = {
        "추세 국면": (trend["phase"], sid["price"]),
        "이동평균 배열": (trend["evidence"][0]["value"], sid["price"]),
        "RSI(14)": (round(float(last["rsi14"]), 1), sid["price"]),
        "거래량(20일 평균 대비 배)": (round(float(last["volume"] / last["vol_ma20"]), 2), sid["price"]),
        "최근 신호": (patterns[0]["type"] if patterns else "없음", sid["price"]),
        "MACD-시그널": (round(float(last["macd"] - last["macd_signal"]), 1), sid["price"]),
        "ADX": (round(float(last["adx14"]), 1), sid["price"]),
        "60일 수익률(%)": (rets.get("return_60d_pct"), sid["price"]),
        "60일선 대비(%)": (round((price / float(last["ma60"]) - 1) * 100, 1), sid["price"]),
        "120일선 대비(%)": (round((price / float(last["ma120"]) - 1) * 100, 1), sid["price"]),
        "지지선(원)": (levels["support"], sid["price"]),
        "저항선(원)": (levels["resistance"], sid["price"]),
        "매물대(원)": (levels["volume_node"], sid["price"]),
        "ATR(14, 원)": (round(float(last["atr14"]), 0), sid["price"]),
        "외국인 20일 순매수(억원)": (flow["foreign_20d_억원"], sid["flow"]),
        "기관 20일 순매수(억원)": (flow["institution_20d_억원"], sid["flow"]),
        "최근 1년 최대낙폭(%)": (risk["mdd_1y_pct"], sid["price"]),
        "하루 평균 움직임(%)": (risk["daily_move_pct"], sid["price"]),
        "연환산 변동성(%)": (risk["volatility_annual_pct"], sid["price"]),
        "베타": (risk["beta"], sid["index"]),
    }

    result = TechnicalResult(
        ticker=ticker,
        name=name_of(ticker),
        as_of=str(ind.index[-1].date()),
        persona_id=persona.id,
        trend=Phase(phase=trend["phase"], evidence=trend["evidence"]),
        signals=[Signal(**{k: v for k, v in p.items() if k != "direction"}) for p in patterns],
        levels=levels,
        flows=flow,
        risk=risk,
        factor_scores=T.factor_scores(ind, flow),
        metrics=pick_metrics(available, PRIORITY.get(persona.id, []), persona),
        chart_annotations=[{"date": p["date"], "label": p["type"]} for p in patterns]
        + [{"price": str(v), "label": "지지선"} for v in levels["support"]]
        + [{"price": str(v), "label": "저항선"} for v in levels["resistance"]],
        sources=sources,
        limitations=["기술적 신호는 과거 통계이며 미래를 보장하지 않음"],
    )
    if any(s.reliability == "참고용" for s in result.signals):
        result.limitations.append(f"표본이 {T.MIN_SAMPLES}건 미만인 신호는 참고용")

    opp, rsk = _rule_items(trend["phase"], patterns, last, risk, flow, levels, sid)
    persona_horizon = req.horizon or persona.horizon
    system = system_prompt(
        "기술적 분석을 가르치는 차트 코치",
        f"{result.name}의 현재 추세, 모멘텀, 수급, 주요 가격대를 진단합니다. 신호를 말할 때는 과거 적중률과 표본 수를 함께 말합니다.",
        persona,
    )
    facts = facts_payload(
        result,
        {"trend": trend, "signals": patterns, "levels": levels, "flows": flow, "risk": risk,
         "returns": rets, "horizon": persona_horizon, "persona_title": persona.title},
        opp, rsk, sources,
    )
    lead = f"{josa(result.name)} 지금 {PHASE_PLAIN[trend['phase']]} 흐름이에요."
    if persona.level == "L1":
        lead += f" 최근 1년 동안 고점에서 최대 {abs(risk['mdd_1y_pct'])}% 떨어진 적이 있어요."
    return finalize(
        result, persona=persona, writer=(writer or Writer()) if use_llm else None, system=system,
        facts=facts, question=req.question, opp=opp, risk=rsk, lead=lead,
        closing="차트 신호는 과거 통계일 뿐 앞으로를 보장하지 않아요.",
        glossary_terms=TERMS.get(persona.level, []),
    )


def _rule_items(phase, patterns, last, risk, flow, levels, sid) -> tuple[list[dict], list[dict]]:
    opp, rsk = [], []
    p = sid["price"]
    if phase == "상승추세":
        opp.append({"text": "주가가 20일선과 60일선 위에 있는 상승추세", "source_id": p})
    elif phase == "하락추세":
        rsk.append({"text": "주가가 20일선과 60일선 아래에 있는 하락추세", "source_id": p})
    elif phase == "변동성 확대":
        rsk.append({"text": "평소보다 하루 움직임이 크게 커진 상태", "source_id": p})
    for s in patterns[:3]:
        if s["hit_rate_20d"] is None:
            continue
        txt = f"{s['date']} {s['type']} 신호 (과거 {s['n_samples']}번 중 20일 뒤 {'상승' if s['direction'] == '상승' else '하락'} 비율 {s['hit_rate_20d']}%)"
        (opp if s["direction"] == "상승" else rsk).append({"text": txt, "source_id": p})
    if last["rsi14"] >= 70:
        rsk.append({"text": f"RSI {round(float(last['rsi14']), 1)}로 단기 과열 구간", "source_id": p})
    elif last["rsi14"] <= 30:
        opp.append({"text": f"RSI {round(float(last['rsi14']), 1)}로 단기 과매도 구간", "source_id": p})
    if risk["mdd_1y_pct"] <= -30:
        rsk.append({"text": f"최근 1년 최대낙폭 {risk['mdd_1y_pct']}%로 하락 폭이 컸던 종목", "source_id": p})
    both = flow["foreign_20d_억원"] + flow["institution_20d_억원"]
    if both > 0:
        opp.append({"text": f"최근 20일 외국인·기관 합산 {round(both, 1)}억원 순매수", "source_id": sid["flow"]})
    elif both < 0:
        rsk.append({"text": f"최근 20일 외국인·기관 합산 {round(abs(both), 1)}억원 순매도", "source_id": sid["flow"]})
    if not opp:
        sup = levels["support"][0] if levels["support"] else None
        opp.append({"text": f"가까운 지지선 {sup:,.0f}원 근처에서 하락이 멈췄던 이력" if sup else "뚜렷한 상승 신호는 아직 없음", "source_id": p})
    if not rsk:
        rsk.append({"text": f"연환산 변동성 {risk['volatility_annual_pct']}%만큼 흔들릴 수 있음", "source_id": p})
    return opp, rsk

### `kic_agents/agents/industry_agent.py`
산업·사업모델 분석 에이전트

In [ ]:
%%writefile kic_agents/agents/industry_agent.py
"""산업·사업모델 분석 에이전트 (설계서 5장).

흐름: 기업 개요(밸류체인 위치) → 재무·수출·동종사 조회 → 업황 국면(C)
→ 사업모델 정리 → 기회·리스크 → 페르소나에 맞춰 지표 선택 → 문장 작성 → 자체 점검
"""

from __future__ import annotations

from kic_agents.agents.base import facts_payload, finalize, josa, pick_metrics
from kic_agents.data.providers import DataProvider
from kic_agents.data.universe import UNIVERSE, name_of, segment_of
from kic_agents.llm import Writer, system_prompt
from kic_agents.personas import get_persona
from kic_agents.schemas import AgentRequest, IndustryResult, Phase
from kic_agents.tools import industry as I

EXPORT_MONTHS = 36

PRIORITY = {
    "L1-AGG": ["밸류체인 위치", "업황 국면"],
    "L1-CON": ["밸류체인 위치", "부채비율(%)"],
    "L2-AGG": ["매출 성장률(YoY, %)", "영업이익률(%)", "반도체 수출 YoY(%)", "업황 국면"],
    "L2-CON": ["부채비율(%)", "영업이익률(%)", "매출 성장률(YoY, %)", "업황 국면"],
    "L3-AGG": ["업황 국면", "반도체 수출 YoY(%)", "재고회전율(회)", "매출 성장률(YoY, %)", "영업이익률(%)",
               "동종사 영업이익률 순위", "최대 고객 매출 비중(%)", "영업이익률 변화(%p)"],
    "L3-CON": ["업황 국면", "설비투자/매출(%)", "부채비율(%)", "영업이익률(%)", "재고회전율(회)",
               "최대 고객 매출 비중(%)", "매출 성장률(YoY, %)", "동종사 영업이익률 순위"],
}
TERMS = {
    "L1": ["밸류체인", "업황 사이클", "부채비율"],
    "L2": ["업황 사이클", "영업이익률", "매출 성장률", "부채비율"],
}

_derived_cache: dict[tuple[int, str], dict[str, dict]] = {}


def universe_derived(provider: DataProvider) -> dict[str, dict]:
    """유니버스 전체 파생지표 (동종사 비교·백분위용). 공급자·기준일별로 캐시."""
    key = (id(provider), str(provider.base_date))
    if key not in _derived_cache:
        _derived_cache[key] = {t: I.derive_financials(provider.get_financials(t)) for t in provider.tickers()}
    return _derived_cache[key]


def run(req: AgentRequest, provider: DataProvider, writer: Writer | None = None, use_llm: bool = True) -> IndustryResult:
    persona = get_persona(req.persona_id)
    ticker = req.ticker
    end = req.as_of or provider.base_date

    profile = provider.get_profile(ticker)
    exports = provider.get_exports(end, EXPORT_MONTHS)
    derived_all = universe_derived(provider)
    derived = derived_all.get(ticker) or I.derive_financials(provider.get_financials(ticker))
    filings = provider.search_filings(ticker, "사업의 내용", k=2)

    sources = [provider.source(k, None if k == "exports" else ticker) for k in ("profile", "financials", "exports", "filing")]
    sid = {"profile": sources[0].id, "financials": sources[1].id, "exports": sources[2].id, "filing": sources[3].id}

    cycle = I.classify_cycle(exports, derived)
    for ev in cycle["evidence"]:
        ev["source_id"] = sid[ev.pop("source")]
    segments = {t: segment_of(t) for t in derived_all}
    names = {t: v["name"] for t, v in UNIVERSE.items()}
    peers = I.compare_peers(ticker, derived_all, segments, names)
    rank = next((i + 1 for i, r in enumerate(peers) if r["is_target"]), None)
    share = profile.get("top_customer_share")

    available = {
        "밸류체인 위치": (f"{profile['segment']} ({profile['products']})", sid["profile"]),
        "업황 국면": (cycle["phase"], sid["exports"]),
        "반도체 수출 YoY(%)": (cycle["evidence"][0]["value"], sid["exports"]),
        "매출 성장률(YoY, %)": (derived["revenue_yoy_pct"], sid["financials"]),
        "영업이익률(%)": (derived["op_margin_pct"], sid["financials"]),
        "영업이익률 변화(%p)": (derived["op_margin_change_pp"], sid["financials"]),
        "재고회전율(회)": (derived["inventory_turnover"], sid["financials"]),
        "부채비율(%)": (derived["debt_ratio_pct"], sid["financials"]),
        "설비투자/매출(%)": (derived["capex_to_revenue_pct"], sid["financials"]),
        "최대 고객 매출 비중(%)": (round(share * 100) if share is not None else None, sid["profile"]),
        "동종사 영업이익률 순위": (f"{len(peers)}개 중 {rank}위" if rank else None, sid["financials"]),
    }
    available = {k: v for k, v in available.items() if v[0] is not None}

    result = IndustryResult(
        ticker=ticker,
        name=profile.get("name") or name_of(ticker),
        as_of=str(end),
        persona_id=persona.id,
        value_chain={"segment": profile["segment"], "detail": profile["products"], "source_id": sid["profile"]},
        business_model={
            "products": profile["products"],
            "customers": profile.get("customers"),
            "top_customer_share_pct": available.get("최대 고객 매출 비중(%)", (None,))[0],
            "moat": profile.get("moat"),
            "capex_intensity_pct": derived["capex_to_revenue_pct"],
            "filing_excerpts": [{"text": f["text"], "title": f["title"], "source_id": sid["filing"]} for f in filings],
        },
        cycle=Phase(phase=cycle["phase"], evidence=cycle["evidence"]),
        peers=peers,
        factor_scores=I.factor_scores(ticker, derived_all),
        metrics=pick_metrics(available, PRIORITY.get(persona.id, []), persona),
        sources=sources,
        limitations=[f"재무 기준 분기: {derived['quarter']}", "업황 국면은 반도체 전체 수출 기준이며 개별 회사 상황과 다를 수 있음"],
    )

    opp, rsk = I.rule_items(derived, cycle, profile, sid)
    system = system_prompt(
        "반도체 산업 애널리스트이자 투자 교육자",
        f"{result.name}의 밸류체인 위치, 사업모델, 업황 국면을 근거와 함께 설명합니다.",
        persona,
    )
    facts = facts_payload(
        result,
        {"value_chain": result.value_chain, "business_model": result.business_model, "cycle": cycle,
         "financials": derived, "peers": peers, "horizon": req.horizon or persona.horizon,
         "persona_title": persona.title},
        opp, rsk, sources,
    )
    if persona.level == "L1":
        lead = f"{josa(result.name)} 반도체를 만드는 과정 중 '{profile['segment']}' 쪽 회사로, {profile['products']} 일을 해요."
    else:
        lead = f"{josa(result.name)} {profile['segment']} 밸류체인에 있고, 반도체 업황은 지금 '{cycle['phase']}' 국면이에요."
    return finalize(
        result, persona=persona, writer=(writer or Writer()) if use_llm else None, system=system,
        facts=facts, question=req.question, opp=opp, risk=rsk, lead=lead,
        closing="업황은 몇 년 주기로 바뀌니, 숫자가 언제 기준인지 꼭 함께 보세요.",
        glossary_terms=TERMS.get(persona.level, []),
    )

### `kic_agents/orchestrator.py`
Orchestrator 연결: 의도 분류, 랭킹, AI 투자위원회, LangGraph 노드

In [ ]:
%%writefile kic_agents/orchestrator.py
"""Orchestrator 연결부 (설계서 7장).

- handle(): 질문 의도에 따라 두 에이전트를 부르고 결과를 합친다.
- rank_universe(): Weight 슬라이더용. LLM 없이 팩터 점수만으로 17종목 랭킹.
- committee(): AI 투자위원회. 같은 점수를 위원 4명이 각자 가중치로 해석.
- industry_node / technical_node: LangGraph 노드로 바로 쓸 수 있는 함수.
"""

from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor

from kic_agents.agents import industry_agent, technical_agent
from kic_agents.data.providers import DataProvider
from kic_agents.data.universe import name_of
from kic_agents.llm import Writer
from kic_agents.personas import get_persona
from kic_agents.schemas import AgentRequest

AGENTS = {"industry_bm": industry_agent.run, "technical": technical_agent.run}

INTENT_KEYWORDS = {
    "committee": ("위원회", "토론"),
    "ranking": ("랭킹", "순위", "추천 종목", "가중치", "weight"),
    "report": ("보고서", "리포트", "종합", "분석해"),
    "technical": ("차트", "추세", "기술적", "이동평균", "rsi", "지지", "저항", "수급", "타이밍", "흐름"),
    "company": ("뭐 하는", "무슨 회사", "사업", "돈 벌", "산업", "업황", "밸류체인", "고객", "경쟁"),
}
ROUTES = {
    "company": ["industry_bm"],
    "technical": ["technical"],
    "report": ["industry_bm", "technical"],
    "committee": ["industry_bm", "technical"],
    "ranking": [],
}


def classify_intent(question: str) -> str:
    """키워드 규칙. 팀의 Orchestrator가 LLM 분류를 쓰면 이 함수를 바꾸면 된다."""
    q = question.lower()
    for intent, words in INTENT_KEYWORDS.items():
        if any(w in q for w in words):
            return intent
    return "report"


def handle(req: AgentRequest, provider: DataProvider, writer: Writer | None = None, intent: str | None = None) -> dict:
    intent = intent or classify_intent(req.question)
    names = ROUTES[intent]
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = {n: pool.submit(AGENTS[n], req, provider, writer) for n in names}
        results = {n: f.result() for n, f in futures.items()}
    scores: dict[str, float] = {}
    for r in results.values():
        scores.update(r.factor_scores)
    out = {
        "intent": intent,
        "ticker": req.ticker,
        "persona_id": req.persona_id,
        "factor_scores": scores,
        "results": {n: r.model_dump() for n, r in results.items()},
    }
    if intent == "committee":
        out["committee"] = committee(scores)
    if intent == "ranking":
        out["ranking"] = rank_universe(provider, req.persona_id, req.weights)
    return out


def factor_scores_for(ticker: str, provider: DataProvider) -> dict[str, float]:
    req = AgentRequest(ticker=ticker, persona_id="L4-CON")
    scores = {}
    scores.update(industry_agent.run(req, provider, use_llm=False).factor_scores)
    scores.update(technical_agent.run(req, provider, use_llm=False).factor_scores)
    return scores


_score_cache: dict[tuple[int, str], dict[str, dict[str, float]]] = {}


def universe_scores(provider: DataProvider) -> dict[str, dict[str, float]]:
    """17종목 팩터 점수. 하루 한 번 계산해 캐시(실서비스에서는 NEON에 저장)."""
    key = (id(provider), str(provider.base_date))
    if key not in _score_cache:
        _score_cache[key] = {t: factor_scores_for(t, provider) for t in provider.tickers()}
    return _score_cache[key]


def weighted_score(scores: dict[str, float], weights: dict[str, float]) -> tuple[float, dict[str, float]]:
    """가중합(0~100)과 팩터별 기여도. 점수가 없는 팩터의 가중치는 나머지에 다시 나눈다."""
    used = {f: w for f, w in weights.items() if f in scores and w > 0}
    total_w = sum(used.values())
    if total_w == 0:
        return 0.0, {}
    contrib = {f: round(scores[f] * w / total_w * 100, 1) for f, w in used.items()}
    return round(sum(contrib.values()), 1), contrib


def rank_universe(provider: DataProvider, persona_id: str, weights: dict[str, float] | None = None) -> list[dict]:
    weights = weights or get_persona(persona_id).default_weights
    rows = []
    for t, s in universe_scores(provider).items():
        total, contrib = weighted_score(s, weights)
        rows.append({"ticker": t, "name": name_of(t), "score": total, "contributions": contrib})
    rows.sort(key=lambda r: r["score"], reverse=True)
    for i, r in enumerate(rows, 1):
        r["rank"] = i
    return rows


# 2차 회의 4대 유형 (네 번째는 팀 확정 전 가안)
COMMITTEE = {
    "안정형 위원": {"stability": 40, "profitability": 30, "value": 20, "momentum": 10},
    "성장형 위원": {"growth": 45, "momentum": 25, "profitability": 15, "flow": 15},
    "유동형 위원": {"momentum": 45, "flow": 40, "sentiment": 15},
    "가치형 위원": {"value": 40, "profitability": 30, "stability": 30},
}


def committee(scores: dict[str, float]) -> list[dict]:
    """위원별 점수와 입장. 매수·매도 표현 대신 관심/중립/신중을 쓴다."""
    out = []
    for member, w in COMMITTEE.items():
        total, contrib = weighted_score(scores, w)
        stance = "관심" if total >= 60 else ("신중" if total < 40 else "중립")
        top = max(contrib, key=contrib.get) if contrib else None
        out.append({"member": member, "score": total, "stance": stance, "main_factor": top, "contributions": contrib})
    return out


# ---------------------------------------------------------------- LangGraph 노드

def _node(agent_name: str, provider: DataProvider):
    def node(state: dict) -> dict:
        req = AgentRequest(**state["request"])
        res = AGENTS[agent_name](req, provider)
        return {"results": {**state.get("results", {}), agent_name: res.model_dump()}}

    node.__name__ = f"{agent_name}_node"
    return node


def industry_node(provider: DataProvider):
    """사용 예: graph.add_node("industry_bm", industry_node(provider))"""
    return _node("industry_bm", provider)


def technical_node(provider: DataProvider):
    return _node("technical", provider)

## 2부. 실행해 보기

코드 셀을 고친 뒤 다시 실행했다면 아래 import 셀부터 다시 실행하세요 (autoreload가 켜져 있어 대부분 자동 반영돼요).

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from kic_agents.schemas import AgentRequest
from kic_agents.personas import all_personas, classify_persona, adjust_level
from kic_agents.data.providers import SampleDataProvider
from kic_agents.data.universe import UNIVERSE
from kic_agents.agents import technical_agent, industry_agent
from kic_agents.tools import technical as T, industry as I
from kic_agents.orchestrator import handle, rank_universe, committee, classify_intent

# 한글 폰트 (Windows: 맑은 고딕, Mac: AppleGothic, 그 외: 나눔고딕이 있으면 사용)
# Colab에서 한글이 네모로 나오면: !apt-get -qq install fonts-nanum 실행 후 런타임 다시 시작
for f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f
        break
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_colwidth", 120)

provider = SampleDataProvider()   # 실제 시세: 맨 아래 pykrx 셀 참고
print("기준일:", provider.base_date, "/ 종목 수:", len(provider.tickers()))

### 2-1. 페르소나 8종과 온보딩 판별 (분류 A)

In [ ]:
pd.DataFrame([
    {"persona_id": p.id, "이름": p.title, "지표 수": p.max_metrics, "용어 풀이": p.explain_terms,
     "요약 문장": f"{p.summary_len[0]}~{p.summary_len[1]}", "강조": p.emphasis, "기간": p.horizon}
    for p in all_personas()
])

In [ ]:
# 이해도 5문항(0~2점) + 성향 5문항(1~5점) → persona_id
print(classify_persona([1, 1, 1, 1, 1], [5, 5, 5, 5, 5]))   # L2-AGG
print(classify_persona([1, 1, 1, 1, 1], [5, 5, 5, 2, 2]))   # 여유 자금이 적으면 안정형 → L2-CON
# 대화 중 용어 수준으로 등급 조정
print(adjust_level("L2-AGG", ["DCF랑 WACC 가정 보여줘", "백테스트 결과는?"]))

### 2-2. 기술적 분석 에이전트

In [ ]:
TICKER, PERSONA = "000660", "L2-AGG"   # 바꿔 보세요

tech = technical_agent.run(AgentRequest(ticker=TICKER, persona_id=PERSONA, question="요즘 차트 어때요?"), provider)
print(f"[{tech.name}] 요약 작성: {tech.summary_by}\n")
print(tech.summary)
pd.DataFrame([m.model_dump() for m in tech.metrics])

In [ ]:
print("추세 국면:", tech.trend.phase)
display(pd.DataFrame([e.model_dump() for e in tech.trend.evidence]))
print("최근 신호와 과거 적중률(20일 뒤)")
display(pd.DataFrame([s.model_dump() for s in tech.signals]) if tech.signals else "최근 20일 신호 없음")
print("기회:", [i.text for i in tech.opportunities])
print("리스크:", [i.text for i in tech.risks])
print("용어:", [(t.term, t.explanation) for t in tech.terms])

In [ ]:
# 차트: 종가, 20·60일선, 지지·저항, 신호
ind = T.calc_indicators(provider.get_ohlcv(TICKER, provider.base_date, 800)).tail(250)
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
ax.plot(ind.index, ind["close"], label="종가", lw=1.2)
ax.plot(ind.index, ind["ma20"], label="20일선", lw=1)
ax.plot(ind.index, ind["ma60"], label="60일선", lw=1)
for v in tech.levels.get("support", []):
    ax.axhline(v, ls="--", lw=0.8, color="tab:green")
for v in tech.levels.get("resistance", []):
    ax.axhline(v, ls="--", lw=0.8, color="tab:red")
for s in tech.signals:
    d = pd.Timestamp(s.date)
    if d in ind.index:
        ax.annotate(s.type, (d, ind.loc[d, "close"]), xytext=(0, 25), textcoords="offset points",
                    ha="center", fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.6))
ax.set_title(f"{tech.name} · {tech.trend.phase} (점선: 초록 지지 / 빨강 저항)")
ax.legend(loc="upper left")
ax2.plot(ind.index, ind["rsi14"], color="tab:purple", lw=1)
ax2.axhline(70, ls=":", color="gray"); ax2.axhline(30, ls=":", color="gray")
ax2.set_ylabel("RSI(14)")
plt.tight_layout(); plt.show()

### 2-3. 산업·사업모델 분석 에이전트

In [ ]:
ind_res = industry_agent.run(AgentRequest(ticker=TICKER, persona_id=PERSONA, question="뭘로 돈 벌어요?"), provider)
print(f"[{ind_res.name}] 요약 작성: {ind_res.summary_by}\n")
print(ind_res.summary)
print("\n밸류체인:", ind_res.value_chain)
print("업황 국면:", ind_res.cycle.phase)
display(pd.DataFrame([e.model_dump() for e in ind_res.cycle.evidence]))
pd.DataFrame(ind_res.peers)

In [ ]:
# 업황 국면 근거: 반도체 수출 YoY
e = I.export_trend(provider.get_exports(provider.base_date, 36)).dropna()
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(e.index, e["yoy_pct"], color=["tab:blue" if v >= 0 else "tab:orange" for v in e["yoy_pct"]])
ax.axhline(0, color="black", lw=0.8)
ax.set_title(f"반도체 수출 YoY(%) · 현재 국면: {ind_res.cycle.phase} (샘플 데이터)")
ax.tick_params(axis="x", rotation=90, labelsize=8)
plt.tight_layout(); plt.show()

### 2-4. 같은 종목, 페르소나 8종 비교
등급이 오르면 지표 수가 늘고 용어 풀이가 줄며, 성향에 따라 기회·리스크 중 먼저 말하는 쪽이 바뀌어요.

In [ ]:
rows = []
for p in all_personas():
    r = technical_agent.run(AgentRequest(ticker=TICKER, persona_id=p.id), provider, use_llm=False)
    rows.append({"persona": p.id, "이름": p.title, "지표": ", ".join(m.name for m in r.metrics[:4]),
                 "용어 풀이 수": len(r.terms), "요약": r.summary})
pd.DataFrame(rows)

### 2-5. Orchestrator: 질문 의도 → 에이전트 호출 → AI 투자위원회

In [ ]:
for q in ["SK하이닉스 요즘 차트 어때요?", "한미반도체는 뭘로 돈 벌어요?", "이 종목 위원회 열어 줘", "삼성전자 보고서 써 줘"]:
    print(f"{q!r:40} → {classify_intent(q)}")

In [ ]:
out = handle(AgentRequest(ticker=TICKER, persona_id="L3-AGG", question="이 종목 위원회 열어 줘"), provider)
print("의도:", out["intent"], "/ 호출된 에이전트:", list(out["results"]))
print("팩터 점수(0~1):", out["factor_scores"])
pd.DataFrame(out["committee"])

### 2-6. Weight 랭킹 (슬라이더용)
가중치를 바꾸면 17종목 순위와 팩터별 기여도가 바로 바뀌어요. 처음 한 번은 17종목 점수를 계산하느라 몇 초 걸려요.

In [ ]:
weights = {"growth": 30, "momentum": 25, "flow": 15, "profitability": 15, "stability": 15}   # 바꿔 보세요
rank = pd.DataFrame(rank_universe(provider, "L2-AGG", weights))
contrib = pd.DataFrame(rank.pop("contributions").tolist()).fillna(0)
table = pd.concat([rank, contrib], axis=1)
table

In [ ]:
top = table.head(10).set_index("name")[contrib.columns]
top.iloc[::-1].plot.barh(stacked=True, figsize=(10, 5), title="상위 10종목 점수와 팩터별 기여도")
plt.xlabel("점수(0~100)"); plt.tight_layout(); plt.show()

In [ ]:
# (선택) ipywidgets가 있으면 슬라이더로 조절:  %pip install ipywidgets
try:
    import ipywidgets as w
    sliders = {f: w.IntSlider(value=v, min=0, max=50, description=f) for f, v in weights.items()}
    def show(**ws):
        r = pd.DataFrame(rank_universe(provider, "L2-AGG", ws)).drop(columns="contributions")
        display(r.head(10))
    w.interact(show, **sliders)
except ImportError:
    print("ipywidgets가 없어 슬라이더를 건너뜁니다. 위 weights 값을 바꿔 다시 실행하세요.")

### 2-7. 팀원에게 넘길 JSON
Orchestrator·프론트 담당은 이 형식만 보면 됩니다.

In [ ]:
print(json.dumps(tech.model_dump(), ensure_ascii=False, indent=2)[:2500], "...")

### 2-8. 자체 점검
숫자 규칙과 페르소나 규칙이 지켜지는지 확인해요. 모두 통과하면 마지막 줄에 OK가 나와요.

In [ ]:
from kic_agents.llm import unknown_numbers
for p in all_personas():
    for run in (technical_agent.run, industry_agent.run):
        r = run(AgentRequest(ticker=TICKER, persona_id=p.id), provider, use_llm=False)
        ids = {s.id for s in r.sources}
        assert r.opportunities and r.risks, p.id
        assert all(i.source_id in ids for i in r.opportunities + r.risks), p.id
        assert p.level == "L4" or len(r.metrics) <= p.max_metrics, p.id
        assert unknown_numbers(r.summary, r.model_dump()) == [], (p.id, r.summary)
print("OK: 8개 페르소나 × 2개 에이전트")

## 3부. (선택) 실제 데이터

- **실제 시세·수급**: `pip install pykrx` 후 아래 셀 실행. 재무·수출·공시는 아직 샘플이에요.
- **NEON**: 데이터 담당이 적재를 마치면 `PostgresProvider(os.environ["DATABASE_URL"])`로 바꾸면 됩니다 (`pip install "psycopg[binary]"`, 테이블 규격은 zip 안 `sql/schema.sql`).

In [ ]:
USE_REAL = False   # True로 바꾸면 실행
if USE_REAL:
    from kic_agents.data.providers import PykrxProvider
    real = PykrxProvider()
    r = technical_agent.run(AgentRequest(ticker="005930", persona_id="L2-CON"), real)
    print(r.as_of, r.trend.phase); print(r.summary)